# Session 03 · Bayesian Networks and their Practical Application

Run each cell with **Shift+Enter**. Original material by Aseef Ahmed.

In [1]:
# Setup: run from the code/ folder so local modules and data files are found
import os
if os.path.isdir('code'):
    os.chdir('code')

## Example 1 - define the network and CPTs (see bn.example_net)

In [2]:
from bn import example_net, learn_cpt, rejection_sample

net = example_net()
for n in net.nodes:
    print(n, "parents:", net.parents[n], "CPT:", net.cpt[n])

Summer parents: [] CPT: {(): 0.4}
UncleTesting parents: [] CPT: {(): 0.05}
LoadShedding parents: ['Summer'] CPT: {(True,): 0.7, (False,): 0.2}
GeneratorOn parents: ['LoadShedding', 'UncleTesting'] CPT: {(True, True): 0.99, (True, False): 0.9, (False, True): 0.95, (False, False): 0.02}
UPSBeeps parents: ['LoadShedding'] CPT: {(True,): 0.8, (False,): 0.05}


## Example 2 - joint probability of one full world

In [3]:
world = {"Summer": True, "UncleTesting": False, "LoadShedding": True, "GeneratorOn": True, "UPSBeeps": True}
print("P(world) =", round(net.joint(world), 5))

P(world) = 0.19152


## Example 3 - diagnostic query by enumeration

In [4]:
print("P(LoadShedding):", round(net.query("LoadShedding", {})[True], 3))
print("P(LoadShedding | GeneratorOn):", round(net.query("LoadShedding", {"GeneratorOn": True})[True], 3))

P(LoadShedding): 0.4
P(LoadShedding | GeneratorOn): 0.901


## Example 4 - explaining away

In [5]:
print("P(LoadShedding | GeneratorOn, UncleTesting):",
      round(net.query("LoadShedding", {"GeneratorOn": True, "UncleTesting": True})[True], 3))

P(LoadShedding | GeneratorOn, UncleTesting): 0.41


## Example 5 - rejection sampling

In [6]:
approx = rejection_sample(net, "LoadShedding", {"GeneratorOn": True}, n=20000, seed=0)
print("sampled estimate:", round(approx[True], 3))

sampled estimate: 0.898


## Example 6 - learn CPTs from data

In [7]:
df = net.sample_df(5000, seed=1)
learned = learn_cpt(df, "GeneratorOn", ["LoadShedding", "UncleTesting"])
for k, v in learned.items():
    print(k, "learned", round(v, 3), "true", net.cpt["GeneratorOn"][k])

(True, True) learned 0.982 true 0.99
(True, False) learned 0.901 true 0.9
(False, True) learned 0.9 true 0.95
(False, False) learned 0.02 true 0.02
